# <B> Create Custom Docker Image </B>
* Container: codna_pytorch_p310

## AutoReload

In [1]:
%load_ext autoreload
%autoreload 2

## 1. For preprocessing

In [2]:
import boto3
from utils.ecr import ecr_handler

In [3]:
ecr = ecr_handler()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")

### 1.1 Get base image uri

In [4]:
import sagemaker

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/pydantic/_internal/_fields.py:172: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[05/30/25 05:12:10] INFO     Found credentials from IAM Role:                                   ]8;id=810166;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=932615;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [5]:
base_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    version="2.1",
    region=region,
    image_scope="training",
    instance_type="ml.g5.xlarge"
)

print (f'base_image_uri: {base_image_uri}')

                    INFO     Defaulting to only available Python version: py310                   ]8;id=505728;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=414342;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py#610\610]8;;\

base_image_uri: 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310


In [6]:
%%writefile ./custom-docker/dockerfile-prep

FROM 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310

RUN pip install -U pip boto3 sagemaker

COPY requirements.txt /opt/ml/packages/
RUN pip install -r /opt/ml/packages/requirements.txt

# 기본 환경변수
ENV PYTHONUNBUFFERED=TRUE

# Processing Job 환경변수
ENV SM_INPUT_DIR=/opt/ml/processing/input
ENV SM_OUTPUT_DIR=/opt/ml/processing/output

Overwriting ./custom-docker/dockerfile-prep


In [7]:
strRepositoryName="prep-docker-image"  ## <-- 원하는 docker repostory 이름을 추가
strRepositoryName = strRepositoryName.lower()
strDockerFile = "dockerfile-prep"
strDockerDir = "./custom-docker/"
strTag = "latest"

* **strAccountId**는 베이스 이미지의 account_id 사용 (**763104351884**.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.6.0-gpu-py312)

In [8]:
ecr.build_docker(strDockerDir, strDockerFile, strRepositoryName, strRegionName=region, strAccountId="763104351884", no_cache=True)
strEcrRepositoryUri_prep = ecr.register_image_to_ecr(region, account_id, strRepositoryName, strTag)

print (f'strEcrRepositoryUri_prep: {strEcrRepositoryUri_prep}')

/home/ec2-user/SageMaker/anomaly-detection-with-explanation
/home/ec2-user/SageMaker/anomaly-detection-with-explanation/custom-docker
strDockerFile dockerfile-prep
aws ecr get-login --region 'us-west-2' --registry-ids '763104351884' --no-include-email


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



#0 building with "default" instance using docker driver

#1 [internal] load build definition from dockerfile-prep
#1 transferring dockerfile: 426B done
#1 DONE 0.0s

#2 [auth] sharing credentials for 763104351884.dkr.ecr.us-west-2.amazonaws.com
#2 DONE 0.0s

#3 [internal] load metadata for 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310
#3 DONE 0.1s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/4] FROM 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310@sha256:4ca33eda180d23556e2317c77e305018d4d1b03c741e0a59032fb3f52f8cd0a7
#5 CACHED

#6 [internal] load build context
#6 transferring context: 38B done
#6 DONE 0.0s

#7 [2/4] RUN pip install -U pip boto3 sagemaker
#7 0.506 Requirement already satisfied: pip in /opt/conda/lib/python3.10/site-packages (24.1.2)
#7 0.620 Collecting pip
#7 0.656   Downloading pip-25.1.1-py3-none-any.whl.metadata (3.6 kB)
#7 0.664 Requirement already satisfied: boto3 in


/home/ec2-user/SageMaker/anomaly-detection-with-explanation
== REGISTER AN IMAGE TO ECR ==
  processing_repository_uri: 615299776985.dkr.ecr.us-west-2.amazonaws.com/prep-docker-image:latest
aws ecr get-login --region 'us-west-2' --registry-ids '615299776985' --no-include-email


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded

aws ecr create-repository --repository-name 'prep-docker-image'



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'prep-docker-image' already exists in the registry with id '615299776985'


docker tag 'prep-docker-image:latest' '615299776985.dkr.ecr.us-west-2.amazonaws.com/prep-docker-image:latest'
docker push '615299776985.dkr.ecr.us-west-2.amazonaws.com/prep-docker-image:latest'
== REGISTER AN IMAGE TO ECR ==
strEcrRepositoryUri_prep: 615299776985.dkr.ecr.us-west-2.amazonaws.com/prep-docker-image:latest


## 2. For training

### 2.1 Get base image uri

In [9]:
base_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    version="2.1",
    region=region,
    image_scope="training",
    instance_type="ml.g5.xlarge"
)

print (f'base_image_uri: {base_image_uri}')

[05/30/25 05:16:38] INFO     Defaulting to only available Python version: py310                   ]8;id=515253;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=753844;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py#610\610]8;;\

base_image_uri: 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310


In [10]:
%%writefile ./custom-docker/dockerfile-tr

FROM 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310

RUN pip install -U pip boto3 sagemaker

COPY requirements.txt /opt/ml/packages/
RUN pip install -r /opt/ml/packages/requirements.txt

# 기본 환경변수
ENV PYTHONUNBUFFERED=TRUE
ENV PATH="/opt/ml/code:${PATH}"
ENV SAGEMAKER_SUBMIT_DIRECTORY /opt/ml/code

Overwriting ./custom-docker/dockerfile-tr


In [11]:
strRepositoryName="tr-docker-image"  ## <-- 원하는 docker repostory 이름을 추가
strRepositoryName = strRepositoryName.lower()
strDockerFile = "dockerfile-tr"
strDockerDir = "./custom-docker/"
strTag = "latest"

* **strAccountId**는 베이스 이미지의 account_id 사용 (**763104351884**.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.6.0-gpu-py312)

In [12]:
ecr.build_docker(strDockerDir, strDockerFile, strRepositoryName, strRegionName=region, strAccountId="763104351884", no_cache=True)
strEcrRepositoryUri_tr = ecr.register_image_to_ecr(region, account_id, strRepositoryName, strTag)

print (f'strEcrRepositoryUri_tr: {strEcrRepositoryUri_tr}')

/home/ec2-user/SageMaker/anomaly-detection-with-explanation
/home/ec2-user/SageMaker/anomaly-detection-with-explanation/custom-docker
strDockerFile dockerfile-tr
aws ecr get-login --region 'us-west-2' --registry-ids '763104351884' --no-include-email


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

#0 building with "default" instance using docker driver

#1 [internal] load build definition from dockerfile-tr


Login Succeeded



#1 transferring dockerfile: 383B done
#1 DONE 0.0s

#2 [auth] sharing credentials for 763104351884.dkr.ecr.us-west-2.amazonaws.com
#2 DONE 0.0s

#3 [internal] load metadata for 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310
#3 DONE 0.1s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [1/4] FROM 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.1-gpu-py310@sha256:4ca33eda180d23556e2317c77e305018d4d1b03c741e0a59032fb3f52f8cd0a7
#5 CACHED

#6 [internal] load build context
#6 transferring context: 38B done
#6 DONE 0.0s

#7 [2/4] RUN pip install -U pip boto3 sagemaker
#7 0.496 Requirement already satisfied: pip in /opt/conda/lib/python3.10/site-packages (24.1.2)
#7 0.614 Collecting pip
#7 0.656   Downloading pip-25.1.1-py3-none-any.whl.metadata (3.6 kB)
#7 0.664 Requirement already satisfied: boto3 in /opt/conda/lib/python3.10/site-packages (1.34.158)
#7 1.177 Collecting boto3
#7 1.188   Downloading boto3-1.38.26


/home/ec2-user/SageMaker/anomaly-detection-with-explanation
== REGISTER AN IMAGE TO ECR ==
  processing_repository_uri: 615299776985.dkr.ecr.us-west-2.amazonaws.com/tr-docker-image:latest
aws ecr get-login --region 'us-west-2' --registry-ids '615299776985' --no-include-email


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded

aws ecr create-repository --repository-name 'tr-docker-image'



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'tr-docker-image' already exists in the registry with id '615299776985'


docker tag 'tr-docker-image:latest' '615299776985.dkr.ecr.us-west-2.amazonaws.com/tr-docker-image:latest'
docker push '615299776985.dkr.ecr.us-west-2.amazonaws.com/tr-docker-image:latest'
== REGISTER AN IMAGE TO ECR ==
strEcrRepositoryUri_tr: 615299776985.dkr.ecr.us-west-2.amazonaws.com/tr-docker-image:latest


## 3. For inference

### 3.1 Get base image uri

In [17]:
base_image_uri = sagemaker.image_uris.retrieve(
    framework="pytorch",
    version="2.1",
    region=region,
    image_scope="inference",
    instance_type="ml.g5.xlarge"
)

print (f'base_image_uri: {base_image_uri}')

[05/30/25 06:56:23] INFO     Defaulting to only available Python version: py310                   ]8;id=914460;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=699803;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/image_uris.py#610\610]8;;\

base_image_uri: 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.1-gpu-py310


In [18]:
%%writefile ./custom-docker/dockerfile-inf

FROM 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.1-gpu-py310

RUN pip install -U pip boto3 sagemaker

COPY requirements.txt /opt/ml/packages/
RUN pip install -r /opt/ml/packages/requirements.txt

# 기본 환경변수
ENV PYTHONUNBUFFERED=TRUE
ENV PATH="/opt/ml/code:${PATH}"
ENV SAGEMAKER_MODEL_DIR=/opt/ml/model

Writing ./custom-docker/dockerfile-inf


In [19]:
strRepositoryName="inf-docker-image"  ## <-- 원하는 docker repostory 이름을 추가
strRepositoryName = strRepositoryName.lower()
strDockerFile = "dockerfile-inf"
strDockerDir = "./custom-docker/"
strTag = "latest"

* **strAccountId**는 베이스 이미지의 account_id 사용 (**763104351884**.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.6.0-gpu-py312)

In [20]:
ecr.build_docker(strDockerDir, strDockerFile, strRepositoryName, strRegionName=region, strAccountId="763104351884", no_cache=True)
strEcrRepositoryUri_inf = ecr.register_image_to_ecr(region, account_id, strRepositoryName, strTag)

print (f'strEcrRepositoryUri_inf: {strEcrRepositoryUri_inf}')

/home/ec2-user/SageMaker/anomaly-detection-with-explanation
/home/ec2-user/SageMaker/anomaly-detection-with-explanation/custom-docker
strDockerFile dockerfile-inf
aws ecr get-login --region 'us-west-2' --registry-ids '763104351884' --no-include-email


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



#0 building with "default" instance using docker driver

#1 [internal] load build definition from dockerfile-inf
#1 transferring dockerfile: 379B done
#1 DONE 0.0s

#2 [internal] load metadata for 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.1-gpu-py310
#2 DONE 0.0s

#3 [internal] load .dockerignore
#3 transferring context: 2B done
#3 DONE 0.0s

#4 [internal] load build context
#4 transferring context: 38B done
#4 DONE 0.0s

#5 [1/4] FROM 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-inference:2.1-gpu-py310
#5 DONE 0.5s

#6 [2/4] RUN pip install -U pip boto3 sagemaker
#6 0.833 Requirement already satisfied: pip in /opt/conda/lib/python3.10/site-packages (24.0)
#6 0.942 Collecting pip
#6 0.979   Downloading pip-25.1.1-py3-none-any.whl.metadata (3.6 kB)
#6 0.987 Requirement already satisfied: boto3 in /opt/conda/lib/python3.10/site-packages (1.28.60)
#6 1.495 Collecting boto3
#6 1.505   Downloading boto3-1.38.26-py3-none-any.whl.metadata (6.6 kB)
#6 1.605 Colle


/home/ec2-user/SageMaker/anomaly-detection-with-explanation
== REGISTER AN IMAGE TO ECR ==
  processing_repository_uri: 615299776985.dkr.ecr.us-west-2.amazonaws.com/inf-docker-image:latest
aws ecr get-login --region 'us-west-2' --registry-ids '615299776985' --no-include-email


WARNING! Using --password via the CLI is insecure. Use --password-stdin.
WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded

aws ecr create-repository --repository-name 'inf-docker-image'
docker tag 'inf-docker-image:latest' '615299776985.dkr.ecr.us-west-2.amazonaws.com/inf-docker-image:latest'
docker push '615299776985.dkr.ecr.us-west-2.amazonaws.com/inf-docker-image:latest'
== REGISTER AN IMAGE TO ECR ==
strEcrRepositoryUri_inf: 615299776985.dkr.ecr.us-west-2.amazonaws.com/inf-docker-image:latest


## 3. [Optional] AWS Systems Manager Parameter Store 를 이용한 파라미터 저장/활용
- [AWS Systems Manager Parameter Store](https://docs.aws.amazon.com/systems-manager/latest/userguide/systems-manager-parameter-store.html)
- Attach IAM polich to sagemaker execution role (<b>with console</b>)
> **SSM**: "arn:aws:iam::aws:policy/AmazonSSMFullAccess"<BR>

In [21]:
from utils.ssm import parameter_store

In [22]:
pm = parameter_store(region)
strPrefix = pm.get_params(key="PREFIX")

In [23]:
pm.put_params(key="-".join([strPrefix, "IMAGE-URI-PREP"]), value=strEcrRepositoryUri_prep, overwrite=True)
pm.put_params(key="-".join([strPrefix, "IMAGE-URI-TR"]), value=strEcrRepositoryUri_tr, overwrite=True)
pm.put_params(key="-".join([strPrefix, "IMAGE-URI-INF"]), value=strEcrRepositoryUri_inf, overwrite=True)

'Store suceess'

In [24]:
print ("IMAGE-URI-PREP: ", pm.get_params(key="-".join([strPrefix, "IMAGE-URI-PREP"])))
print ("IMAGE-URI-TR: ", pm.get_params(key="-".join([strPrefix, "IMAGE-URI-TR"])))
print ("IMAGE-URI-INF: ", pm.get_params(key="-".join([strPrefix, "IMAGE-URI-INF"])))

IMAGE-URI-PREP:  615299776985.dkr.ecr.us-west-2.amazonaws.com/prep-docker-image:latest
IMAGE-URI-TR:  615299776985.dkr.ecr.us-west-2.amazonaws.com/tr-docker-image:latest
IMAGE-URI-INF:  615299776985.dkr.ecr.us-west-2.amazonaws.com/inf-docker-image:latest
